# Crawlling Berita

In [ ]:
"""
Crawler berita Detik untuk kategori nasional dan internasional.

Fitur:
- Ambil artikel dari kategori nasional & internasional
- Looping ke banyak halaman indeks
- Simpan ke CSV & JSON
"""

import requests
from bs4 import BeautifulSoup
import time
import csv
import json

# Konfigurasi
HEADERS = {
    'User-Agent': 'CrawlerBeritaDetik/1.0 (+https://example.com)'
}
SLEEP_BETWEEN_REQUESTS = 1.0
TIMEOUT = 10
MAX_PAGES = 20   # jumlah halaman indeks yang akan dilalui (bisa ditambah)

# ------------------ UTIL ------------------

def safe_get(url):
    try:
        r = requests.get(url, headers=HEADERS, timeout=TIMEOUT)
        r.raise_for_status()
        return r
    except Exception as e:
        print(f"Gagal mengambil {url}: {e}")
        return None

def extract_article_basic(soup):
    title = soup.title.string.strip() if soup.title else None
    paragraphs = [p.get_text(strip=True) for p in soup.find_all('p')]
    body = '\n\n'.join([p for p in paragraphs if p])
    return title, body

# ------------------ DETIK ------------------

def crawl_detik(tag="nasional", max_articles=200):
    base_url = "https://news.detik.com/indeks"
    results = []
    page = 1

    while len(results) < max_articles and page <= MAX_PAGES:
        url = f"{base_url}?tag={tag}&page={page}"
        resp = safe_get(url)
        if not resp:
            break
        soup = BeautifulSoup(resp.text, 'html.parser')

        links = []
        for a in soup.find_all('a', href=True):
            href = a['href']
            if 'detik.com' in href and ('/read/' in href or '/berita/' in href):
                links.append(href)
        links = list(dict.fromkeys(links))

        for link in links:
            if len(results) >= max_articles:
                break
            print("Detik ->", link)
            time.sleep(SLEEP_BETWEEN_REQUESTS)
            r = safe_get(link)
            if not r:
                continue
            s = BeautifulSoup(r.text, 'html.parser')
            title = s.select_one('h1').get_text(strip=True) if s.select_one('h1') else None
            published = s.select_one('div.detail__date').get_text(strip=True) if s.select_one('div.detail__date') else None
            content = s.select_one('div.detail__body-text')
            body = None
            if content:
                paragraphs = [p.get_text(strip=True) for p in content.find_all('p')]
                body = '\n\n'.join([p for p in paragraphs if p])
            else:
                title, body = extract_article_basic(s)

            results.append({
                'source': 'detik',
                'kategori': tag,
                'url': link,
                'title': title,
                'published': published,
                'body': body
            })
        page += 1
    return results

# ------------------ SAVE ------------------

def save_csv(records, filename='berita_detik.csv'):
    if not records:
        return
    keys = list(records[0].keys())
    with open(filename, 'w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(records)
    print("CSV tersimpan:", filename)

def save_json(records, filename='berita_detik.json'):
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(records, f, ensure_ascii=False, indent=2)
    print("JSON tersimpan:", filename)

# ------------------ MAIN ------------------
if __name__ == '__main__':
    records = []
    records += crawl_detik(tag="nasional", max_articles=200)
    records += crawl_detik(tag="internasional", max_articles=200)

    save_csv(records, 'berita_detik_nasional_internasional.csv')
    save_json(records, 'berita_detik_nasional_internasional.json')
    print("Total artikel:", len(records))

Detik -> https://news.detik.com/berita/d-8141021/1-jenazah-santri-korban-runtuhan-ponpes-sidoarjo-dievakuasi-dalam-posisi-sujud
Detik -> https://news.detik.com/berita/d-8141018/siswi-di-bandung-barat-meninggal-diduga-keracunan-dinkes-tak-terkait-mbg
Detik -> https://news.detik.com/berita/d-8140924/jawab-kebingungan-umat-ini-klasifikasi-produk-boikot-israel-menurut-ulama
Detik -> https://news.detik.com/berita/d-8141016/bejat-duo-kakek-kembar-di-bekasi-lecehkan-wanita-disabilitas
Detik -> https://news.detik.com/berita/d-8140970/menkum-resmi-teken-sk-kepengurusan-ppp-ketum-mardiono
Detik -> https://news.detik.com/berita/d-8140967/kuasa-hukum-sebut-staf-ahli-kemensos-edi-suharto-jadi-tersangka-kpk
Detik -> https://news.detik.com/berita/d-8140964/pihak-eks-dosen-uin-malang-tolak-surat-pengusiran-yang-dibuat-warga
Detik -> https://news.detik.com/berita/d-8140931/fpti-kecam-aksi-kekerasan-senior-komunitas-pecinta-alam-di-bitung
Detik -> https://news.detik.com/berita/d-8140930/hr-v-diduga-mela